# Pulizia del file CRM – `hotel_bookings_crm.csv`

Ogni passo di pulizia: **cosa**, **perché**, **righe prima → dopo**.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

RAW_PATH   = '../data/hotel_bookings_crm.csv'
CLEAN_PATH = '../data/crm_clean.csv'


In [2]:
# Il file usa ';' come separatore. Leggiamo tutto come testo (dtype=str)
# così nessuna colonna viene convertita male in automatico.
raw = pd.read_csv(RAW_PATH, sep=';', dtype=str)
df = raw.copy()     # 'raw' resta intatto per i confronti prima/dopo
print(df.shape)
df.head()


(24608, 18)


,BookingRef,property,checkin_date,n_adults,lead_time_days,guest_country,market_segment,distribution_channel,customer_type,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,agent_id,company_id,days_in_waiting_list,reservation_status,reservation_status_date,total_price
0,BK310511,City Hotel,2016-12-03,2,73,USA,Online TA,TA/TO,Transient,0,0,0,7,NaN,0,Check-Out,2016-12-04,NaN
1,BK300310,City Hotel,2016-05-29,2,106 days,DEU,Offline TA/TO,TA/TO,Transient-Party,0,0,0,27,NaN,0,Check-Out,2016-06-01,"242,25"
2,BK302934,City Hotel,2017-07-13,3,356,NOR,Online TA,TA/TO,Transient,0,0,0,9,NaN,0,Canceled,2016-08-29,"529,2"
3,BK320441,Resort Hotel,2016-10-30,2,14,FRA,Online TA,TA/TO,Transient,0,0,0,241,NaN,0,Check-Out,2016-11-14,"799,8"
4,BK316173,City Hotel,2016-04-29,3,27,ITA,Online TA,TA/TO,Transient,0,0,0,9,NaN,0,Canceled,2016-04-04,"531,0"


## 2. Descrizione e report di qualità (prima della pulizia)


In [3]:
# Panoramica: tipo di ogni colonna e quanti valori non mancanti ci sono
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 24608 entries, 0 to 24607
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   BookingRef                      24600 non-null  str  
 1   property                        24600 non-null  str  
 2   checkin_date                    24600 non-null  str  
 3   n_adults                        24354 non-null  str  
 4   lead_time_days                  24600 non-null  str  
 5   guest_country                   24600 non-null  str  
 6   market_segment                  24600 non-null  str  
 7   distribution_channel            24600 non-null  str  
 8   customer_type                   24600 non-null  str  
 9   is_repeated_guest               24600 non-null  str  
 10  previous_cancellations          24600 non-null  str  
 11  previous_bookings_not_canceled  24600 non-null  str  
 12  agent_id                        21280 non-null  str  
 13  company_id  

In [4]:
# Quanti valori mancano in ogni colonna, in numero e in percentuale
missing = df.isna().sum().to_frame('n_missing')
missing['pct'] = (missing['n_missing'] / len(df) * 100).round(2)
missing


,n_missing,pct
BookingRef,8,0.03
property,8,0.03
checkin_date,8,0.03
n_adults,254,1.03
lead_time_days,8,0.03
guest_country,8,0.03
market_segment,8,0.03
distribution_channel,8,0.03
customer_type,8,0.03
is_repeated_guest,8,0.03


In [5]:
# Lo stesso valore è scritto in modi diversi? (maiuscole, spazi, errori di battitura)
for c in ['property', 'market_segment', 'distribution_channel', 'customer_type', 'reservation_status']:
    print(f'--- {c} ---')
    print(df[c].value_counts(dropna=False), '\n')


--- property ---
property
City Hotel        15692
Resort Hotel       7924
City Hotel          173
CITY HOTEL          166
 City Hotel         163
city hotel          148
Resort Hotel         87
resort hotel         86
 Resort Hotel        82
RESORT HOTEL         79
NaN                   8
Name: count, dtype: int64 

--- market_segment ---
market_segment
Online TA        11592
Offline TA/TO     4846
Groups            4040
Direct            2484
Corporate         1080
Complementary      140
Online T.A.         81
Onlne TA            57
online ta           49
Aviation            47
Offline TA-TO       31
Ofline TA/TO        24
offline ta/to       24
Groups              23
Direct              18
direct              15
Group               15
groups              13
Corporte             9
NaN                  8
corporate            7
complementary        3
Undefined            2
Name: count, dtype: int64 

--- distribution_channel ---
distribution_channel
TA/TO        20217
Direct        2923

In [6]:
print('Fully duplicated rows:', df.duplicated().sum())
print('Repeated BookingRef:', df['BookingRef'].dropna().duplicated().sum())
print('Fully empty rows:', df.isna().all(axis=1).sum())



Fully duplicated rows: 107
Repeated BookingRef: 100
Fully empty rows: 8


In [7]:
# checkin_date è scritta in più formati: contiamo quanti per tipo
d = df['checkin_date'].dropna()
print('yyyy-mm-dd :', d.str.fullmatch(r'\d{4}-\d{2}-\d{2}').sum())
print('dd/mm/yyyy :', d.str.fullmatch(r'\d{2}/\d{2}/\d{4}').sum())
print('dd Mon yyyy:', d.str.fullmatch(r'\d{1,2} [A-Za-z]{3} \d{4}').sum())


yyyy-mm-dd : 19867
dd/mm/yyyy : 3687
dd Mon yyyy: 1046


In [8]:
# lead_time_days dovrebbe essere solo un numero, ma alcuni valori sono tipo "106 days"
lead = df['lead_time_days'].dropna()
print('Non-numeric lead_time values:', (~lead.str.fullmatch(r'\d+')).sum())
print('Examples:', lead[~lead.str.fullmatch(r'\d+')].head().tolist())

# total_price usa la virgola come decimale
print('total_price values with a comma:', df['total_price'].dropna().str.contains(',').sum())


Non-numeric lead_time values: 982
Examples: ['106 days', '86 days', '1 days', '214 days', '186 days']
total_price values with a comma: 24106


## 3. Cleaning


### 3.1 Whitespace and empty rows
- **Cosa:** togliamo gli spazi all'inizio e alla fine di ogni cella ed eliminiamo le righe completamente vuote.
- **Perché:** `' City Hotel'` e `'City Hotel  '` sono lo stesso valore; le righe vuote non contengono nessuna prenotazione.


In [9]:
# Togliamo gli spazi all'inizio e alla fine di ogni cella (tutte le colonne sono testo)
df = df.apply(lambda s: s.str.strip())

# Eliminiamo le righe completamente vuote
n = len(df)
df = df.dropna(how='all')
print(f'Empty rows: {n} -> {len(df)} rows ({n - len(df)} removed)')


Empty rows: 24608 -> 24600 rows (8 removed)


### 3.2 Duplicates
- **Cosa:** eliminiamo le righe identiche.
- **Perché:** ogni `BookingRef` ripetuto è una copia esatta della stessa prenotazione; tenerla gonfierebbe i conteggi e i ricavi.
- Dopo questo passo `BookingRef` deve essere unico, perché è la chiave per il merge.


In [10]:
# Eliminiamo le righe identiche (stessa prenotazione esportata due volte)
n = len(df)
df = df.drop_duplicates()
print(f'Duplicates: {n} -> {len(df)} rows ({n - len(df)} removed)')

# Controllo: BookingRef deve essere unico e mai vuoto (serve per il merge)
assert df['BookingRef'].notna().all()
assert df['BookingRef'].is_unique
print('BookingRef is unique:', df['BookingRef'].is_unique)


Duplicates: 24600 -> 24500 rows (100 removed)
BookingRef is unique: True


### 3.3 Categorical columns
- **Cosa:** riportiamo ogni variante (maiuscole/minuscole, errori di battitura, trattini) a un unico valore corretto.
- **Perché:** altrimenti un `groupby` conta `Canceled`, `CANCELED` e `Cancelled` come tre stati diversi.
- **`Undefined`** in `market_segment` e `distribution_channel` è un segnaposto per "non noto", non una categoria reale: è impossibile già da solo, quindi diventa NaN. Prima di cambiarlo si creano i flag `segment_undefined` e `channel_undefined`. Nessuna riga viene eliminata.

In [11]:
# VERIFICA: quante righe hanno 'Undefined', prima di decidere (nessuna modifica)
print("market_segment = 'Undefined':", (df['market_segment'].str.lower() == 'undefined').sum())
print("distribution_channel = 'Undefined':", (df['distribution_channel'] == 'Undefined').sum())

market_segment = 'Undefined': 2
distribution_channel = 'Undefined': 3


In [12]:
# property: 10 varianti -> 2 valori ('City Hotel', 'Resort Hotel')
df['property'] = df['property'].str.title()
df['property'].value_counts(dropna=False)


property
City Hotel      16283
Resort Hotel     8217
Name: count, dtype: int64

In [13]:
# market_segment: prima tutto minuscolo, poi correggiamo gli errori di battitura
seg = df['market_segment'].str.lower()
seg = seg.replace({
    'online t.a.': 'online ta', 'onlne ta': 'online ta',
    'offline ta-to': 'offline ta/to', 'ofline ta/to': 'offline ta/to',
    'group': 'groups', 'corporte': 'corporate',
})
# Flag prima di cambiare: 'undefined' diventerà NaN
df['segment_undefined'] = seg == 'undefined'

# poi riportiamo ogni valore alla scrittura ufficiale ('undefined' -> mancante)
df['market_segment'] = seg.map({
    'online ta': 'Online TA', 'offline ta/to': 'Offline TA/TO', 'groups': 'Groups',
    'direct': 'Direct', 'corporate': 'Corporate', 'complementary': 'Complementary',
    'aviation': 'Aviation', 'undefined': np.nan,
})
print('Segments set to NaN:', df['segment_undefined'].sum())
df['market_segment'].value_counts(dropna=False)

Segments set to NaN: 2


market_segment
Online TA        11737
Offline TA/TO     4907
Groups            4071
Direct            2504
Corporate         1090
Complementary      142
Aviation            47
NaN                  2
Name: count, dtype: int64

In [14]:
# minuscolo e senza spazi/trattini: 'Check-Out', 'CheckOut', 'check out' diventano tutti 'checkout'
status = df['reservation_status'].str.lower().str.replace(r'[\s-]', '', regex=True)
df['reservation_status'] = status.map({
    'checkout': 'Check-Out', 'canceled': 'Canceled', 'cancelled': 'Canceled', 'noshow': 'No-Show',
})
# Controllo: una variante non presente nella mappa diventerebbe NaN
assert df['reservation_status'].notna().all(), 'Unmapped status variant'
df['reservation_status'].value_counts(dropna=False)


reservation_status
Check-Out    15326
Canceled      8918
No-Show        256
Name: count, dtype: int64

In [15]:
# 'Undefined' non è un canale reale -> valore mancante (prima il flag)
df['channel_undefined'] = df['distribution_channel'] == 'Undefined'
df['distribution_channel'] = df['distribution_channel'].replace('Undefined', np.nan)
print('Channels set to NaN:', df['channel_undefined'].sum())
df['distribution_channel'].value_counts(dropna=False)

Channels set to NaN: 3


distribution_channel
TA/TO        20140
Direct        2905
Corporate     1402
GDS             50
NaN              3
Name: count, dtype: int64

### 3.4 Country codes
- **Cosa:** tutto maiuscolo; `PT` e `Portugal` → `PRT` (stesso paese scritto in modo diverso); `unknown` e `-` → NaN.
- **Perché:** `unknown` e `-` sono segnaposto: `isna()` non li vede, ma non sono paesi reali.
- **`CN` non si cambia.** È ambiguo: la Cina compare già come `CHN` (217 righe), mentre il Canada (`CAN`) non compare mai, quindi `CN` è probabilmente il Canada. Non è provato riga per riga e il PMS non ha il paese, quindi il valore resta `CN` e le righe si segnano con il flag `country_cn_ambiguous`.
- Prima di cambiare si crea il flag `country_placeholder`. Nessuna riga eliminata.

In [16]:
# VERIFICA: valori che non sono codici a 3 lettere, e i codici vicini a 'CN' (nessuna modifica)
c = df['guest_country'].str.upper()
print("'CHN':", (c == 'CHN').sum(), "| 'CAN':", (c == 'CAN').sum())
c[c.str.len() != 3].value_counts(dropna=False)

'CHN': 217 | 'CAN': 0


guest_country
CN          228
UNKNOWN      61
-            53
PT           23
PORTUGAL     23
Name: count, dtype: int64

In [17]:
# Flag prima di cambiare (e flag per 'CN', che resta invariato)
paese = df['guest_country'].str.upper()
df['country_placeholder'] = paese.isin(['UNKNOWN', '-'])
df['country_cn_ambiguous'] = paese == 'CN'

# Uniformiamo i codici paese; i segnaposto diventano NaN; 'CN' non si tocca
df['guest_country'] = paese.replace({
    'PT': 'PRT', 'PORTUGAL': 'PRT',
    'UNKNOWN': np.nan, '-': np.nan,
})
print('Country placeholders set to NaN:', df['country_placeholder'].sum())
print("'CN' flagged as ambiguous (unchanged):", df['country_cn_ambiguous'].sum())

# Controllo: quali valori non sono lunghi 3 lettere?
c = df['guest_country']
print('Non 3-letter codes:', c[c.str.len() != 3].value_counts(dropna=False).to_dict())

Country placeholders set to NaN: 114
'CN' flagged as ambiguous (unchanged): 228
Non 3-letter codes: {'CN': 228, nan: 114}


### 3.5 Dates
- **Cosa:** `checkin_date` ha 3 formati (`2016-12-03`, `03/12/2016`, `03 Dec 2016`): li leggiamo uno alla volta con `pd.to_datetime(..., errors='coerce')` e uniamo i risultati con `fillna`.
- **Perché:** servono date vere (`datetime64`) per usare `.dt` e confrontare le date tra loro.


In [18]:
# Ogni formato viene letto a parte: ciò che non corrisponde diventa NaT
iso  = pd.to_datetime(df['checkin_date'], format='%Y-%m-%d', errors='coerce')
eu   = pd.to_datetime(df['checkin_date'], format='%d/%m/%Y', errors='coerce')
text = pd.to_datetime(df['checkin_date'], format='%d %b %Y', errors='coerce')
df['checkin_date'] = iso.fillna(eu).fillna(text)

df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'], format='%Y-%m-%d')

# Controllo: nessuna data deve restare non letta
assert df['checkin_date'].notna().all(), 'Unparsed check-in dates'
print('Check-in range:', df['checkin_date'].min().date(), '->', df['checkin_date'].max().date())


Check-in range: 2015-07-01 -> 2017-08-31


### 3.6 Numeric columns
- **Cosa:** togliamo `" days"` da `lead_time_days`, sostituiamo la virgola decimale con il punto in `total_price`, convertiamo in numero le colonne intere.
- **Perché:** con il testo non si possono fare somme o medie (`'1650' + '980'` = `'1650980'`).
- Le colonne con NaN (`n_adults`, `agent_id`, `company_id`) diventano `float64`: NaN è un float.


In [19]:
df['lead_time_days'] = pd.to_numeric(df['lead_time_days'].str.replace(' days', '', regex=False))
df['total_price']    = pd.to_numeric(df['total_price'].str.replace(',', '.', regex=False))

num_cols = ['n_adults', 'is_repeated_guest', 'previous_cancellations',
            'previous_bookings_not_canceled', 'agent_id', 'company_id', 'days_in_waiting_list']
for col in num_cols:
    df[col] = pd.to_numeric(df[col])

# Prima di correggere guardiamo min e max (describe mette in vista gli estremi impossibili)
df[['n_adults', 'lead_time_days', 'total_price', 'days_in_waiting_list']].describe()


,n_adults,lead_time_days,total_price,days_in_waiting_list
count,24256.000000,24500.000000,24010.000000,24500.000000
mean,1.858550,113.810490,359.156165,2.267102
std,0.615799,333.236553,336.733011,17.430131
min,0.000000,0.000000,0.000000,0.000000
25%,2.000000,18.000000,148.985000,0.000000
50%,2.000000,70.000000,270.000000,0.000000
75%,2.000000,160.000000,450.000000,0.000000
max,55.000000,9999.000000,5400.000000,391.000000


### 3.7 Accuracy: impossible values
**VERIFICA** (cella sotto): quanti sono i valori sospetti, prima di decidere.

**Decisioni:**
- **`lead_time_days = 9999`**: è un segnaposto per "non noto", non un'attesa di 27 anni. È impossibile già da solo → NaN, con `.mask()`. Prima si crea il flag `lead_time_missing`.
- **`n_adults` oltre 10** (26 e 55): **nessuna modifica in questa fase.** Il valore non è impossibile da solo, perché esistono gruppi veri (entrambe le righe sono `Group`). Si segna con `adults_over_10`; nella Fase B si confronta con `adults` del PMS, come previsto nel notebook del PMS.
- **`total_price`:** la regola IQR segnala circa 1.500 prezzi alti, ma sono soggiorni lunghi o gruppi (*"IQR flags long tails"*): sono valori reali, quindi si tengono.

In [20]:
# VERIFICA: una maschera per ogni controllo (nessuna modifica)
lead_placeholder = df['lead_time_days'] == 9999
print('lead_time = 9999:', lead_placeholder.sum(),
      '| highest other value:', df.loc[~lead_placeholder, 'lead_time_days'].max())

molti = df['n_adults'] > 10
print('n_adults > 10:', molti.sum(), '| values:', df.loc[molti, 'n_adults'].tolist(),
      '| customer_type:', df.loc[molti, 'customer_type'].tolist())

# total_price: regola IQR solo per misurare
q1, q3 = df['total_price'].quantile([0.25, 0.75])
high = q3 + 1.5 * (q3 - q1)
print(f'total_price above IQR fence ({high:.0f}):', (df['total_price'] > high).sum())
print('Negative prices:', (df['total_price'] < 0).sum())

lead_time = 9999: 25 | highest other value: 737
n_adults > 10: 2 | values: [26.0, 55.0] | customer_type: ['Group', 'Group']
total_price above IQR fence (902): 1504
Negative prices: 0


In [21]:
# lead_time = 9999 -> NaN (prima il flag)
df['lead_time_missing'] = df['lead_time_days'] == 9999
df['lead_time_days'] = df['lead_time_days'].mask(df['lead_time_missing'])
print('lead_time set to NaN:', df['lead_time_missing'].sum())

# n_adults > 10: nessuna modifica, solo il flag per la Fase B
df['adults_over_10'] = df['n_adults'] > 10
print('n_adults > 10 flagged (unchanged):', df['adults_over_10'].sum())

lead_time set to NaN: 25
n_adults > 10 flagged (unchanged): 2


### 3.8 Consistency between columns
- **Cosa:** una prenotazione `Check-Out` non può avere la data di stato *prima* del check-in.
- **Mossa scelta:** *flag*, perché non sappiamo quale delle due date sia sbagliata. Aggiungiamo la colonna `date_inconsistent` e decidiamo dopo il confronto col PMS.


In [22]:
# Regola tra colonne: check-out prima del check-in = impossibile
date_bad = (df['reservation_status'] == 'Check-Out') & \
           (df['reservation_status_date'] < df['checkin_date'])
df['date_inconsistent'] = date_bad
print('Check-Out dated before check-in:', date_bad.sum())


Check-Out dated before check-in: 37


## 4. Export


In [23]:
# DATASET PROFILE: crm_clean.csv
# Rows x columns : 24,500 x 25 (raw: 24,608 x 18)
# Removed        : 8 empty rows, 100 exact duplicates
# Fixed          : spellings (property, segment, status, country), 3 date formats,
#                  ' days' in lead_time, decimal comma in total_price
# Set to NaN     : 25 lead_time = 9999, 114 country placeholders,
#                  2 'Undefined' segments, 3 'Undefined' channels
# Flagged only   : 228 'CN' (China or Canada? left unchanged), 2 n_adults > 10,
#                  37 Check-Out dated before check-in
# Open questions : n_adults > 10 and missing prices -> check against the PMS in Phase B

print('Raw shape:  ', raw.shape)
print('Clean shape:', df.shape)
print('\nMissing values after cleaning:')
print(df.isna().sum()[df.isna().sum() > 0])

# Quante righe segna ogni flag
flags = ['segment_undefined', 'channel_undefined', 'country_placeholder', 'country_cn_ambiguous',
         'lead_time_missing', 'adults_over_10', 'date_inconsistent']
print('\nRows marked by each flag:')
print(df[flags].sum())

df.to_csv(CLEAN_PATH, index=False)
print(f'\nSaved clean CRM to {CLEAN_PATH}')

Raw shape:   (24608, 18)
Clean shape: (24500, 25)

Missing values after cleaning:
n_adults                  244
lead_time_days             25
guest_country             114
market_segment              2
distribution_channel        3
agent_id                 3300
company_id              23124
total_price               490
dtype: int64

Rows marked by each flag:
segment_undefined         2
channel_undefined         3
country_placeholder     114
country_cn_ambiguous    228
lead_time_missing        25
adults_over_10            2
date_inconsistent        37
dtype: int64



Saved clean CRM to ../data/crm_clean.csv
